# 01 - Ingesta Bronze: RUES (Registro Mercantil)

Trabajo Práctico 1 - Sección 3 (Fuente 1): Pipeline de ingesta PySpark

## Fuente

* **Dataset**: Personas Naturales, Personas Jurídicas y Entidades Sin Ánimo de Lucro (RUES)
* **API**: `https://www.datos.gov.co/resource/c82u-588k.json` (Socrata / SoQL)
* **Volumen total en la fuente**: 9.407.309 registros (verificado con `$select=count(*)`)
* **Destino**: `Datos_Empresas.bronze.rues_registro_mercantil`

## Estrategia de carga

RUES se trata como una **foto (snapshot) recurrente** del estado del registro mercantil: cada ejecución sobrescribe la tabla Bronze con el estado más reciente de las empresas filtradas. La carga **incremental obligatoria** del trabajo se demuestra en la Fuente 2 (`02_Ingesta_Bronze_TRM.ipynb`).

Dado que el dataset completo supera los 9 millones de registros (impráctico de descargar por HTTP paginado en una sesión de clase), se filtra por `fecha_actualizacion` para traer solo lo actualizado en los **últimos 2 años**. Esto ya supera ampliamente el mínimo de 20.000-50.000 registros exigido, y el filtro se puede ampliar o quitar fácilmente cambiando `FECHA_CORTE`.

## Regla de inmutabilidad (Capa Bronze)

* No se renombran columnas: se conservan exactamente los nombres que entrega la API (`codigo_camara`, `razon_social`, etc.).
* No se hace *casting* destructivo: todas las columnas de RUES son de tipo `text` en el origen, así que se dejan como texto (no se convierten fechas a `DATE` ni números a `INT`/`DOUBLE`). Esa limpieza es tarea de la futura Capa Silver.
* Solo se agregan las columnas de auditoría obligatorias: `_ingested_at` y `_source`.

In [ ]:
%python
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F

BASE_URL = "https://www.datos.gov.co/resource/c82u-588k.json"
LIMIT = 50000  # tamaño de página soportado por Socrata

# fecha_actualizacion llega como texto 'YYYY/MM/DD HH:MM:SS...', comparable como string
FECHA_CORTE = (datetime.now() - timedelta(days=365 * 2)).strftime("%Y/%m/%d")
WHERE_CLAUSE = f"fecha_actualizacion >= '{FECHA_CORTE}'"

print(f"Fuente: {BASE_URL}")
print(f"Filtro aplicado: {WHERE_CLAUSE}")

---

## Paso 1: Contar registros disponibles con el filtro aplicado

In [ ]:
%python
def contar_registros(where=None):
    params = {"$select": "count(*)"}
    if where:
        params["$where"] = where
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    return int(resp.json()[0]["count"])


total_registros = contar_registros(WHERE_CLAUSE)
print(f"Total de registros de RUES actualizados desde {FECHA_CORTE}: {total_registros:,}")

---

## Paso 2: Descargar los datos paginando (HTTP GET + `$limit`/`$offset`)

In [ ]:
%python
MAX_REGISTROS = total_registros


def descargar_datos(max_registros, where=None):
    registros = []
    offset = 0

    while offset < max_registros:
        pagina_limit = min(LIMIT, max_registros - offset)
        params = {"$limit": pagina_limit, "$offset": offset}
        if where:
            params["$where"] = where
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        pagina = resp.json()

        if not pagina:
            break

        registros.extend(pagina)
        offset += LIMIT
        print(f"Descargados {len(registros)} de {max_registros} registros...")

    return registros


registros = descargar_datos(MAX_REGISTROS, WHERE_CLAUSE)
df_pandas = pd.DataFrame(registros)
print(f"\nDataset descargado: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")
df_pandas.head()

---

## Paso 3: Convertir a Spark DataFrame (sin transformar tipos) y agregar columnas de auditoría

In [ ]:
%python
# Se respeta el tipo con el que Socrata entrega cada campo (todo texto en este dataset).
# No se hace ningún .astype()/cast manual: eso sería un casteo destructivo, prohibido en Bronze.
df_spark = spark.createDataFrame(df_pandas)

df_bronze = (
    df_spark
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit(BASE_URL))
)

df_bronze.printSchema()
display(df_bronze.limit(10))

---

## Paso 4: Persistir en Delta Lake (Capa Bronze)

In [ ]:
%python
TABLA_DESTINO = "Datos_Empresas.bronze.rues_registro_mercantil"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_DESTINO)
)

print(f"Tabla Delta escrita: {TABLA_DESTINO}")

In [ ]:
%sql
-- Verificación rápida de la ingesta
DESCRIBE EXTENDED Datos_Empresas.bronze.rues_registro_mercantil;

In [ ]:
%sql
SELECT COUNT(*) AS total_filas, MIN(_ingested_at) AS primera_carga, MAX(_ingested_at) AS ultima_carga
FROM Datos_Empresas.bronze.rues_registro_mercantil;

SELECT * FROM Datos_Empresas.bronze.rues_registro_mercantil LIMIT 10;

---

## Siguiente paso

Continuar con [`02_Ingesta_Bronze_TRM.ipynb`](02_Ingesta_Bronze_TRM.ipynb) para la fuente complementaria con carga incremental.